In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
Economic_Bleed = pd.read_csv('Economic_Bleed.csv')
Economic_Bleed.head()

In [ ]:
# Group by Category and sum the loss
category_loss = Economic_Bleed.groupby('category_name')['production_loss_usd'].sum().sort_values(ascending=False).reset_index()
category_loss.head()

In [ ]:
# Limiting the view to the Top 10, the "Signal" becomes much clearer than the "Noise"
category_loss = category_loss.head(10).copy()

In [ ]:
#Calculating Cumulative Percentage
category_loss['cum_percent'] = 100 * (category_loss['production_loss_usd'].cumsum() / category_loss['production_loss_usd'].sum())
category_loss.head()

In [ ]:
# Top 10 category names
top_10_names = category_loss.head(10)['category_name'].tolist()
top_10_names

In [ ]:
# ROOT CAUSE ANALYSIS 
top_10_details = Economic_Bleed[Economic_Bleed['category_name'].isin(top_10_names)]
top_10_details

In [ ]:
# Pivot Table correctly using the detailed dataframe
pivot_delay = pd.pivot_table(
    top_10_details,
    index='category_name', 
    columns='shipping_mode', 
    values='days_late', 
    aggfunc='mean'
).fillna(0)
pivot_delay

In [ ]:
# VISUALIZATION 
# Plot - Pareto Analysis
fig, ax1 = plt.subplots(figsize=(14,6))
ax1.bar(category_loss['category_name'], category_loss['production_loss_usd'], color='steelblue', label='Financial Loss')
ax2 = ax1.twinx()
ax2.plot(category_loss['category_name'], category_loss['cum_percent'], color='red', marker='D', label='Cumulative %')

ax1.set_ylabel('Production Loss (USD)', fontsize=12)
ax2.set_ylabel('Cumulative Percentage (%)', fontsize=12)
plt.xticks(rotation=45)
plt.title('Pareto Analysis: Financial Bleed by Category', fontsize=14, fontweight='bold')
plt.show()



In [ ]:
# Plot - Root Cause Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_delay, annot=True, cmap='YlOrRd', fmt='.1f')
plt.title('Root Cause: Avg Days Late by Shipping Mode for Top 10 Categories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(pivot_delay)

In [ ]:
# 1. Define the Cost of Change
# Industry average: Upgrading from Second Class to First Class costs ~$5.00 per item
UPGRADE_COST_PER_ORDER = 5.00 

# 2. Filter for the "Second Class" bottleneck in Top 10 categories
bottleneck_orders = top_10_details[top_10_details['shipping_mode'] == 'Second Class'].copy()

# 3. Calculate "Current Loss" vs "Projected Loss"
# We know Second Class averages 2.5 days late, while First Class averages 1.0 day
current_total_loss = bottleneck_orders['production_loss_usd'].sum()

# Estimate projected loss by scaling the delay down to 1.0 day
# (Projected Loss = Current Loss * (Target Delay / Current Delay))
bottleneck_orders['projected_loss'] = bottleneck_orders['production_loss_usd'] * (1.0 / 2.5)
projected_total_loss = bottleneck_orders['projected_loss'].sum()

# 4. Calculate ROI
total_upgrade_cost = len(bottleneck_orders) * UPGRADE_COST_PER_ORDER
gross_savings = current_total_loss - projected_total_loss
net_profit = gross_savings - total_upgrade_cost

print(f"--- Financial Recovery Simulation ---")
print(f"Current Financial Bleed (Second Class): ${current_total_loss:,.2f}")
print(f"Projected Bleed after Upgrade:          ${projected_total_loss:,.2f}")
print(f"Gross Savings:                         ${gross_savings:,.2f}")
print(f"Total Cost to Upgrade Shipping:        ${total_upgrade_cost:,.2f}")
print(f"---")
print(f"NET PROFIT RECOVERY:                   ${net_profit:,.2f}")